# Poster2Movie — 01. Clasificador: cartel → géneros

Versión **base**: CLIP ViT-B/32 congelado + una capa lineal + un umbral por género (detalles en `p2m/clasificador.py`).

**Entorno: GPU (T4)** si hay cuota; si no, también corre en CPU (más lento). `Entorno de ejecución → Cambiar tipo de entorno de ejecución`.

## Preparar el entorno

Monta tu Google Drive (ahí vive el dataset, `poster2movie_datos/`), clona el repo e instala lo necesario. Si el dataset te lo compartieron, agrega la carpeta `poster2movie_datos` a *Mi unidad* como acceso directo.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["P2M_DATOS"] = "/content/drive/MyDrive/poster2movie_datos"
# EQUIPOS: el dataset compartido es solo de lectura; guarden sus modelos y resultados en una
# carpeta suya quitando el # de la siguiente línea (cambien "equipoX" por su equipo):
# os.environ["P2M_SALIDAS"] = "/content/drive/MyDrive/poster2movie_equipoX"
SALIDAS = os.environ.get("P2M_SALIDAS", os.environ["P2M_DATOS"])
REPO = "AndreaHortaHernandez/poster2movie"
if not os.path.isdir("/content/repo"):
    !git clone https://github.com/{REPO}.git /content/repo
else:
    !cd /content/repo && git pull
%cd /content/repo
!pip install -q -r requirements.txt

## 1. Vectores CLIP de todos los carteles

Una sola vez: ~20-30 min con GPU, ~1.5-2 h en CPU (también funciona sin GPU). Se guardan en `poster2movie_datos/embeddings_clip_b32.npz`; si Colab se desconecta, al volver a correrla continúa donde se quedó, y si ya existen no hace nada.

In [ ]:
if not os.path.exists(os.environ["P2M_DATOS"] + "/embeddings_clip_b32.npz"):
    !python -m p2m.clasificador embeddings
else:
    print("Los vectores ya estaban calculados.")

## 2. Entrenar la capa lineal (segundos)

In [ ]:
!python -m p2m.clasificador entrenar

## 3. Evaluar en test

F1 macro es la métrica principal (pesa igual a todos los géneros). Se compara contra una línea base trivial. Los resultados quedan en Drive: `poster2movie_datos/resultados/clasificador/`.

In [ ]:
!python -m p2m.evaluar_clasificador

## 4. Probar con un cartel

In [ ]:
from google.colab import files
from PIL import Image
from p2m.clasificador import Clasificador
from p2m.config import GENEROS_ES

clasificador = Clasificador()
for nombre in files.upload():
    resultado = clasificador.predecir(Image.open(nombre).convert('RGB'))
    print(nombre, "->", ", ".join(GENEROS_ES[g] for g in resultado["generos"]))
    top = sorted(resultado["probabilidades"].items(), key=lambda x: -x[1])[:5]
    print("   ", ", ".join(f"{GENEROS_ES[g]} {p:.2f}" for g, p in top))